# **TUẦN 41 — TÍCH HỢP ATTENTION-FUSION VÀO PIPELINE UNIFASHION**


## **1 — Kiểm tra GPU**

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), (
    "Chưa có GPU/CUDA. Vào Runtime → Change runtime type → GPU."
)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4


## **2 — Clone đúng hai repo**

In [1]:
from pathlib import Path
import subprocess

def run(cmd):
    subprocess.run(cmd, check=True)

PROJECT = Path("/content/khoaluantotnghiep")
UPSTREAM = Path("/content/UniFashion-upstream")

if not PROJECT.exists():
    run(["git", "clone", "--branch", "hien",
         "https://github.com/okok0814/khoaluantotnghiep.git",
         str(PROJECT)])

if not UPSTREAM.exists():
    run(["git", "clone",
         "https://github.com/xiangyu-mm/UniFashion.git",
         str(UPSTREAM)])

assert (PROJECT / "scripts/train_unifashion_finetune.py").exists(), \
    "Sai repo hoặc thiếu script train."
assert (PROJECT / "src/unifashion_runtime.py").exists(), \
    "Thiếu unifashion_runtime.py."

# Tương thích với script cũ dùng đường dẫn /content/UniFashion
alias = Path("/content/UniFashion")
if not alias.exists():
    alias.symlink_to(UPSTREAM, target_is_directory=True)

print("PROJECT:", PROJECT)
print("UPSTREAM:", UPSTREAM)

PROJECT: /content/khoaluantotnghiep
UPSTREAM: /content/UniFashion-upstream


## **3 — Mount Google Drive**

In [3]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/khoaluantotnghiep")
UPSTREAM_DIR = Path("/content/UniFashion-upstream")

DATASET_DIR = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset"
)
IMAGES_DIR = DATASET_DIR / "images"
IMAGE_SPLITS_DIR = DATASET_DIR / "image_splits"
CAPTIONS_DIR = DATASET_DIR / "captions"

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("UPSTREAM_DIR:", UPSTREAM_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("IMAGE_SPLITS_DIR:", IMAGE_SPLITS_DIR)
print("CAPTIONS_DIR:", CAPTIONS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR: /content/khoaluantotnghiep
UPSTREAM_DIR: /content/UniFashion-upstream
DATASET_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset
IMAGES_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/images
IMAGE_SPLITS_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/image_splits
CAPTIONS_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/captions
OUTPUT_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults


## **4 — Kiểm tra checkpoint và dữ liệu**

In [5]:
# Kiểm tra dataset không quét từng ảnh
required_dirs = {
    "dataset": DATASET_DIR,
    "images": IMAGES_DIR,
    "image_splits": IMAGE_SPLITS_DIR,
    "captions": CAPTIONS_DIR,
}

for name, path in required_dirs.items():
    print(f"{'OK' if path.is_dir() else 'MISSING'} | {name}: {path}")

missing_dirs = [
    name for name, path in required_dirs.items()
    if not path.is_dir()
]
assert not missing_dirs, f"Thiếu thư mục dataset: {missing_dirs}"

# Chỉ tìm model assets ở những vị trí đã dự kiến
ASSETS_DIR = Path("/content/drive/MyDrive/UniFashionAssets")

required_assets = [
    ASSETS_DIR / "eva_vit_g.pth",
    ASSETS_DIR / "pretrain" / "none_lora_0.pt",
    ASSETS_DIR / "bert-base-uncased",
]

print("\nMODEL ASSETS")
for path in required_assets:
    print(f"{'OK' if path.exists() else 'MISSING'} | {path}")

print("\nDataset directories PASS")
print("Nếu model assets còn MISSING thì cần sửa ASSETS_DIR trước khi train.")

OK | dataset: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset
OK | images: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/images
OK | image_splits: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/image_splits
OK | captions: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/captions

MODEL ASSETS
MISSING | /content/drive/MyDrive/UniFashionAssets/eva_vit_g.pth
MISSING | /content/drive/MyDrive/UniFashionAssets/pretrain/none_lora_0.pt
MISSING | /content/drive/MyDrive/UniFashionAssets/bert-base-uncased

Dataset directories PASS
Nếu model assets còn MISSING thì cần sửa ASSETS_DIR trước khi train.


## **5 — Kiểm thử Attention-Gated Fusion**

In [18]:
import os

os.chdir(PROJECT)

subprocess.run(
    ["python", "-u", "scripts/test_attention_fusion_adapter.py"],
    check=True
)

CompletedProcess(args=['python', '-u', 'scripts/test_attention_fusion_adapter.py'], returncode=0)

## **6 — Kiểm tra script fine-tuning**

In [7]:
os.chdir(PROJECT)

subprocess.run(
    ["python", "scripts/train_unifashion_finetune.py", "--help"],
    check=True
)

CompletedProcess(args=['python', 'scripts/train_unifashion_finetune.py', '--help'], returncode=0)

## **7 — Chuẩn bị dữ liệu và thử chạy fine-tuning**

In [17]:
from pathlib import Path
import subprocess
import sys
import json

PROJECT = Path("/content/khoaluantotnghiep")
UPSTREAM = Path("/content/UniFashion-upstream")
ASSETS = Path("/content/drive/MyDrive/UniFashionAssets")
DATA_CANDIDATES = [
    Path("/content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashioniq_dataset"),
    Path("/content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset"),
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.is_dir()), None)

CONFIG = PROJECT / "configs/unifashion_finetune_colab.json"
BASE_OUTPUT = Path("/content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults")
OUTPUT = BASE_OUTPUT.with_name("UniFashionResults_run2")

PREPARED_DIR = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared"
)
BUNDLE = PREPARED_DIR.with_suffix(".zip")

def run_command(cmd, cwd):
    print("\nChạy lệnh:", " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout or "(Không có output)", flush=True)
    print("Exit code:", result.returncode, flush=True)
    return result.returncode

# Kiểm tra đường dẫn và môi trường
required_paths = {
    "PROJECT": PROJECT,
    "UPSTREAM": UPSTREAM,
    "ASSETS": ASSETS,
    "CONFIG": CONFIG,
}
missing = [name for name, path in required_paths.items() if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Thiếu đường dẫn: "
        + ", ".join(f"{name}={required_paths[name]}" for name in missing)
    )

if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục FashionIQ. Các thư mục đã kiểm tra:\n"
        + "\n".join(map(str, DATA_CANDIDATES))
    )

IMAGES_DIR = DATA_DIR / "images"
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {IMAGES_DIR}")

print("DATA_DIR:", DATA_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("CONFIG:", CONFIG)
print("OUTPUT:", OUTPUT)

# Chuẩn bị dữ liệu nếu thiếu audit hoặc các file bắt buộc
audit_file = DATA_DIR / "data_audit.json"

if not audit_file.is_file():
    print(
        "\nChưa có data_audit.json. Bắt đầu chuẩn bị dữ liệu...",
        flush=True,
    )

    prep_script = PROJECT / "scripts/prepare_unifashion_finetune.py"
    if not prep_script.is_file():
        raise FileNotFoundError(f"Không tìm thấy script: {prep_script}")

    # Script chuẩn bị dùng exist_ok=False; nếu thư mục cũ còn sót,
    # dùng tên mới để tránh lỗi ghi đè.
    if PREPARED_DIR.exists():
        from datetime import datetime
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        PREPARED_DIR = PREPARED_DIR.with_name(
            f"FashionIQ_prepared_{stamp}"
        )
        BUNDLE = PREPARED_DIR.with_suffix(".zip")

    prep_cmd = [
        sys.executable, "-u", str(prep_script),
        "--images-dir", str(IMAGES_DIR),
        "--annotations-dir", str(DATA_DIR),
        "--upstream", str(UPSTREAM),
        "--csv-dir", str(PROJECT / "data/processed"),
        "--output", str(PREPARED_DIR),
        "--bundle", str(BUNDLE),
    ]

    prep_code = run_command(prep_cmd, PROJECT)

    if prep_code != 0:
        raise RuntimeError(
            "Chuẩn bị dữ liệu thất bại. Chưa chạy fine-tuning. "
            "Xem lỗi ở output phía trên."
        )

    # Chỉ tiếp tục nếu script đã tạo audit ở DATA_DIR.
    # Nếu audit được tạo trong PREPARED_DIR thay vì DATA_DIR,
    # cần điều chỉnh --data-dir theo cấu trúc output thực tế.
    if not audit_file.is_file():
        candidate_audits = [
            PREPARED_DIR / "data_audit.json",
            PROJECT / "data/processed/data_audit.json",
        ]
        found = [p for p in candidate_audits if p.is_file()]
        raise FileNotFoundError(
            "Script chuẩn bị kết thúc nhưng không tạo "
            f"{audit_file}. Audit tìm thấy ở: {found or 'không có'}.\n"
            "Cần xác định thư mục dữ liệu đầu ra đúng trước khi train."
        )

# Kiểm tra JSON audit đọc được
try:
    audit_data = json.loads(audit_file.read_text(encoding="utf-8"))
except Exception as e:
    raise RuntimeError(f"data_audit.json không hợp lệ: {e}") from e

print("\nĐã tìm thấy và đọc được:", audit_file)

# Chạy fine-tuning
train_script = PROJECT / "scripts/train_unifashion_finetune.py"
cmd = [
    sys.executable, "-u", str(train_script),
    "--config", str(CONFIG),
    "--upstream", str(UPSTREAM),
    "--assets", str(ASSETS),
    "--data-dir", str(DATA_DIR),
    "--images-dir", str(IMAGES_DIR),
    "--output", str(OUTPUT),
]

return_code = run_command(cmd, PROJECT)

if return_code != 0:
    print("\nTRAIN FAILED")
else:
    print(
        "\nProcess exit code 0. Hãy kiểm tra log, checkpoint "
        "và trạng thái training trước khi kết luận đã train xong."
    )

DATA_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset
IMAGES_DIR: /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/images
CONFIG: /content/khoaluantotnghiep/configs/unifashion_finetune_colab.json
OUTPUT: /content/drive/MyDrive/KHOALUAN/Tuần 41/UniFashionResults_run2

Chưa có data_audit.json. Bắt đầu chuẩn bị dữ liệu...

Chạy lệnh: /usr/bin/python3 -u /content/khoaluantotnghiep/scripts/prepare_unifashion_finetune.py --images-dir /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/images --annotations-dir /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset --upstream /content/UniFashion-upstream --csv-dir /content/khoaluantotnghiep/data/processed --output /content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared_20261013_163716 --bundle /content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared_20261013_163716.zip


KeyboardInterrupt: 

## **8 — Kiểm tra file dữ liệu đầu vào**

In [15]:

from pathlib import Path

PROJECT = Path("/content/khoaluantotnghiep")
UPSTREAM = Path("/content/UniFashion-upstream")
DATA_DIR = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset"
)

checks = {
    "Train triplets CSV": PROJECT / "data/processed/fashioniq_triplets_train.csv",
    "Val triplets CSV": PROJECT / "data/processed/fashioniq_triplets_val.csv",
    "Upstream train captions": UPSTREAM / "dataset/next_llava.train.dress.caption.json",
    "Upstream val captions": UPSTREAM / "dataset/next_llava.val.dress.caption.json",
    "FashionIQ train dress captions": DATA_DIR / "captions/cap.dress.train.json",
    "FashionIQ val dress captions": DATA_DIR / "captions/cap.dress.val.json",
    "FashionIQ train dress split": DATA_DIR / "image_splits/split.dress.train.json",
    "FashionIQ val dress split": DATA_DIR / "image_splits/split.dress.val.json",
}

for label, path in checks.items():
    print(f"{'OK' if path.is_file() else 'MISSING'} | {label} | {path}")

print("\nCSV files available:")
csv_dir = PROJECT / "data/processed"
if csv_dir.exists():
    for p in sorted(csv_dir.glob("*.csv")):
        print(" ", p.name)
else:
    print("Missing directory:", csv_dir)

print("\nUpstream dataset directory:")
upstream_dataset = UPSTREAM / "dataset"
if upstream_dataset.exists():
    for p in sorted(upstream_dataset.glob("*caption.json")):
        print(" ", p.name)
else:
    print("Missing directory:", upstream_dataset)

OK | Train triplets CSV | /content/khoaluantotnghiep/data/processed/fashioniq_triplets_train.csv
OK | Val triplets CSV | /content/khoaluantotnghiep/data/processed/fashioniq_triplets_val.csv
OK | Upstream train captions | /content/UniFashion-upstream/dataset/next_llava.train.dress.caption.json
OK | Upstream val captions | /content/UniFashion-upstream/dataset/next_llava.val.dress.caption.json
OK | FashionIQ train dress captions | /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/captions/cap.dress.train.json
OK | FashionIQ val dress captions | /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/captions/cap.dress.val.json
OK | FashionIQ train dress split | /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/image_splits/split.dress.train.json
OK | FashionIQ val dress split | /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/image_splits/split.dress.val.json

CSV files available:
  fashioniq_triplets_test.csv
  fashioniq_

## **9 — Chuẩn bị dữ liệu FashionIQ**

In [16]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/content/khoaluantotnghiep")
UPSTREAM = Path("/content/UniFashion-upstream")

DATA_DIR = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset"
)
IMAGES_DIR = DATA_DIR / "images"

PREPARED = Path(
    "/content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared"
)

cmd = [
    sys.executable, "-u",
    str(PROJECT / "scripts/prepare_unifashion_finetune.py"),
    "--images-dir", str(IMAGES_DIR),
    "--annotations-dir", str(DATA_DIR),
    "--upstream", str(UPSTREAM),
    "--csv-dir", str(PROJECT / "data/processed"),
    "--output", str(PREPARED),
    "--bundle", str(PREPARED.parent / "FashionIQ_prepared.zip"),
]

print("Command:", " ".join(cmd), flush=True)
print("=" * 70, flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
print(f"\nPreparation exit code: {return_code}", flush=True)

if return_code == 0:
    required = [
        "train.csv",
        "val.csv",
        "monitor.csv",
        "images_manifest.json",
        "data_audit.json",
    ]
    print("\nKiểm tra đầu ra:")
    for name in required:
        p = PREPARED / name
        print(f"{'OK' if p.is_file() else 'MISSING'} | {p}")
else:
    print("PREPARATION FAILED — gửi t Traceback cuối log.")

Command: /usr/bin/python3 -u /content/khoaluantotnghiep/scripts/prepare_unifashion_finetune.py --images-dir /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset/images --annotations-dir /content/drive/MyDrive/KHOALUAN/Tuần 37/FashionIQ/fashionIQ_dataset --upstream /content/UniFashion-upstream --csv-dir /content/khoaluantotnghiep/data/processed --output /content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared --bundle /content/drive/MyDrive/KHOALUAN/Tuần 41/FashionIQ_prepared.zip


KeyboardInterrupt: 

## **10 — Kiểm tra script đánh giá**

In [ ]:
subprocess.run([
    "python",
    str(PROJECT / "scripts/evaluate_unifashion_fashioniq.py"),
    "--help"
], cwd=PROJECT, check=True)